# 01 — Dataset Exploration

**DRISHTI** — AI-powered marine debris detection from side-scan sonar imagery (SIH 2026, PS 26057)

This notebook explores the primary training dataset (Watertank marine debris FLS),
visualises sample sonar images and masks, analyses class distributions, and assesses
data quality before training.

**Run on:** Colab (CPU is fine) or local Jupyter.

In [ ]:
# ---- Environment setup ----
!pip install -q h5py matplotlib numpy opencv-python-headless scikit-image

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from pathlib import Path
from collections import Counter

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

# Class names matching configs/drishti.yaml
WATERTANK_CLASSES = {
    0: 'background',
    1: 'bottle', 2: 'can', 3: 'chain', 4: 'drink_carton',
    5: 'hook', 6: 'propeller', 7: 'shampoo_bottle',
    8: 'standing_bottle', 9: 'tire', 10: 'valve', 11: 'wall'
}

# Color map for visualisation
CMAP = plt.cm.get_cmap('tab20', len(WATERTANK_CLASSES))
print('Setup complete.')

## 1. Load Dataset

The Watertank dataset (mvaldenegro/marine-debris-fls-datasets) is distributed as
HDF5 files containing sonar images and pixel-level segmentation masks.

In [ ]:
# ---- Download dataset (if not already present) ----
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

HDF5_URL = (
    'https://github.com/mvaldenegro/marine-debris-fls-datasets/'
    'releases/download/watertank-v1.0/watertank_segmentation.h5'
)
HDF5_PATH = DATA_DIR / 'watertank_segmentation.h5'

if not HDF5_PATH.exists():
    print(f'Downloading Watertank HDF5 (~200MB)...')
    !wget -q -O {HDF5_PATH} {HDF5_URL}
    print('Done.')
else:
    print(f'Dataset already present: {HDF5_PATH}')

print(f'File size: {HDF5_PATH.stat().st_size / 1e6:.1f} MB')

In [ ]:
import h5py

hf = h5py.File(HDF5_PATH, 'r')

print('HDF5 keys:', list(hf.keys()))
for key in hf.keys():
    ds = hf[key]
    print(f'  {key}: shape={ds.shape}, dtype={ds.dtype}')

# Extract arrays
images = hf['images'][:] if 'images' in hf else hf[list(hf.keys())[0]][:]
masks = hf['masks'][:] if 'masks' in hf else hf[list(hf.keys())[1]][:]

print(f'\nImages: {images.shape} ({images.dtype})')
print(f'Masks:  {masks.shape} ({masks.dtype})')
print(f'Unique mask values: {np.unique(masks)}')

## 2. Visualise Samples

Display sonar images alongside their ground-truth segmentation masks.

In [ ]:
def show_samples(images, masks, indices=None, n=6):
    """Display sonar images with their segmentation masks."""
    if indices is None:
        indices = np.random.choice(len(images), n, replace=False)
    
    fig, axes = plt.subplots(2, len(indices), figsize=(4 * len(indices), 8))
    if len(indices) == 1:
        axes = axes.reshape(-1, 1)
    
    for col, idx in enumerate(indices):
        img = images[idx]
        msk = masks[idx]
        
        # Handle different shapes (CHW vs HWC)
        if img.ndim == 3 and img.shape[0] in (1, 3):
            img = img[0] if img.shape[0] == 1 else img.transpose(1, 2, 0)
        if msk.ndim == 3:
            msk = msk[0] if msk.shape[0] == 1 else msk[:, :, 0]
        
        # Image
        axes[0, col].imshow(img, cmap='gray')
        axes[0, col].set_title(f'Image #{idx}', fontsize=10)
        axes[0, col].axis('off')
        
        # Mask
        axes[1, col].imshow(msk, cmap=CMAP, vmin=0, vmax=len(WATERTANK_CLASSES) - 1)
        axes[1, col].set_title(f'Mask #{idx}', fontsize=10)
        axes[1, col].axis('off')
    
    # Legend
    unique_vals = np.unique(masks[indices])
    patches = [mpatches.Patch(color=CMAP(v), label=WATERTANK_CLASSES.get(v, f'cls_{v}'))
               for v in unique_vals if v != 0]
    fig.legend(handles=patches, loc='lower center', ncol=min(6, len(patches)),
               fontsize=9, frameon=True)
    
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.08)
    plt.show()

show_samples(images, masks, n=6)

## 3. Class Distribution

Understanding class imbalance is critical — rare classes will need synthetic augmentation.

In [ ]:
# Count pixels per class across the entire dataset
pixel_counts = Counter()
image_counts = Counter()  # how many images contain each class

for i in range(len(masks)):
    msk = masks[i]
    if msk.ndim == 3:
        msk = msk[0] if msk.shape[0] == 1 else msk[:, :, 0]
    
    unique, counts = np.unique(msk, return_counts=True)
    for cls_val, cnt in zip(unique, counts):
        if cls_val == 0:  # skip background
            continue
        pixel_counts[cls_val] += cnt
        image_counts[cls_val] += 1

# Sort by pixel count
sorted_classes = sorted(pixel_counts.keys())
class_names = [WATERTANK_CLASSES.get(c, f'cls_{c}') for c in sorted_classes]
px_counts = [pixel_counts[c] for c in sorted_classes]
img_counts = [image_counts[c] for c in sorted_classes]

print(f'Total images: {len(images)}')
print(f'Total debris classes: {len(sorted_classes)}')
print()
print(f'{"Class":20s} {"Pixel Count":>12s} {"Images":>8s} {"% of pixels":>12s}')
print('-' * 56)
total_debris_px = sum(px_counts)
for name, px, img in zip(class_names, px_counts, img_counts):
    print(f'{name:20s} {px:12,d} {img:8d} {px / total_debris_px * 100:11.2f}%')

In [ ]:
# Visualise class distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

colors = [CMAP(c) for c in sorted_classes]

# Pixel count bar chart
bars1 = ax1.barh(class_names, px_counts, color=colors)
ax1.set_xlabel('Total Pixels')
ax1.set_title('Pixel Distribution by Class')
ax1.invert_yaxis()
for bar, val in zip(bars1, px_counts):
    ax1.text(bar.get_width() + max(px_counts) * 0.01, bar.get_y() + bar.get_height() / 2,
             f'{val:,}', va='center', fontsize=8)

# Image count bar chart
bars2 = ax2.barh(class_names, img_counts, color=colors)
ax2.set_xlabel('Number of Images')
ax2.set_title('Image Occurrence by Class')
ax2.invert_yaxis()
for bar, val in zip(bars2, img_counts):
    ax2.text(bar.get_width() + max(img_counts) * 0.01, bar.get_y() + bar.get_height() / 2,
             str(val), va='center', fontsize=8)

plt.tight_layout()
plt.show()

# Highlight imbalance
max_cls = class_names[np.argmax(px_counts)]
min_cls = class_names[np.argmin(px_counts)]
ratio = max(px_counts) / max(min(px_counts), 1)
print(f'\n⚠ Class imbalance ratio: {ratio:.0f}:1 ({max_cls} vs {min_cls})')
print('→ Synthetic augmentation (build_synthetic_data.py) will help balance rare classes.')

## 4. Data Quality Assessment

Check image resolution, intensity distributions, and noise characteristics.

In [ ]:
# Image statistics
print('Image Statistics:')
sample = images[0]
if sample.ndim == 3 and sample.shape[0] in (1, 3):
    sample = sample[0]
print(f'  Resolution: {sample.shape}')
print(f'  Value range: [{images.min()}, {images.max()}]')
print(f'  Mean intensity: {images.mean():.2f}')
print(f'  Std intensity:  {images.std():.2f}')
print(f'  Dtype: {images.dtype}')

# Intensity histogram (aggregated)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Overall histogram
flat = images.flatten()
if flat.max() <= 1.0:
    flat = (flat * 255).astype(np.uint8)
ax1.hist(flat[::100], bins=128, color='steelblue', alpha=0.7, density=True)
ax1.set_xlabel('Pixel Intensity')
ax1.set_ylabel('Density')
ax1.set_title('Overall Intensity Distribution')

# Per-image mean histogram
means = []
for i in range(len(images)):
    img = images[i]
    if img.ndim == 3:
        img = img[0] if img.shape[0] == 1 else img
    means.append(img.mean())

ax2.hist(means, bins=50, color='coral', alpha=0.7)
ax2.set_xlabel('Mean Intensity')
ax2.set_ylabel('Count')
ax2.set_title('Per-Image Mean Intensity')
ax2.axvline(np.mean(means), color='red', linestyle='--', label=f'μ={np.mean(means):.1f}')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Noise level estimation (using Laplacian variance)
noise_levels = []
for i in range(min(len(images), 200)):
    img = images[i]
    if img.ndim == 3:
        img = img[0] if img.shape[0] == 1 else img
    if img.dtype != np.uint8:
        if img.max() <= 1.0:
            img = (img * 255).astype(np.uint8)
        else:
            img = img.astype(np.uint8)
    laplacian_var = cv2.Laplacian(img, cv2.CV_64F).var()
    noise_levels.append(laplacian_var)

print(f'Noise Level (Laplacian variance):')
print(f'  Mean: {np.mean(noise_levels):.1f}')
print(f'  Std:  {np.std(noise_levels):.1f}')
print(f'  Range: [{np.min(noise_levels):.1f}, {np.max(noise_levels):.1f}]')
print()
if np.mean(noise_levels) > 500:
    print('→ High noise level — Lee speckle filter (preprocess_sonar.py) is important.')
else:
    print('→ Moderate noise level — preprocessing will still help standardise inputs.')

## 5. Summary & Next Steps

| Metric | Value |
|--------|-------|
| Total images | (see above) |
| Debris classes | 11 |
| Resolution | (see above) |
| Key challenge | Class imbalance → synthetic augmentation needed |

**Next notebook:** `02_baseline_training.ipynb` — YOLOv8s-seg fine-tune on Colab/Kaggle GPU.

In [ ]:
hf.close()
print('Dataset exploration complete. Proceed to 02_baseline_training.ipynb')